# 💳 Credit Card Fraud Detection
## Step 1 — Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib
import os

print('Libraries loaded successfully!')

### 1. Load & Explore Data

In [ ]:
def load_data(path):
    df = pd.read_csv(path)
    return df

df = load_data('creditcard.csv')

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Check missing values
print('Missing values:', df.isnull().sum().sum())

# Class distribution
class_counts = df['Class'].value_counts()
print('\nClass Distribution:')
print(f'  Legitimate (0): {class_counts[0]:,} ({class_counts[0]/len(df)*100:.2f}%)')
print(f'  Fraud      (1): {class_counts[1]:,} ({class_counts[1]/len(df)*100:.2f}%)')

In [ ]:
# Visualize class imbalance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(['Legitimate', 'Fraud'], class_counts.values, color=['steelblue', 'crimson'])
axes[0].set_title('Class Distribution (Raw Count)', fontsize=13)
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Amount distribution by class
df[df['Class']==0]['Amount'].hist(bins=50, ax=axes[1], alpha=0.6, label='Legitimate', color='steelblue')
df[df['Class']==1]['Amount'].hist(bins=50, ax=axes[1], alpha=0.8, label='Fraud', color='crimson')
axes[1].set_title('Transaction Amount Distribution', fontsize=13)
axes[1].set_xlabel('Amount ($)')
axes[1].legend()
axes[1].set_xlim(0, 2000)

plt.tight_layout()
plt.savefig('plots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 2. Preprocess — Scale & Split

In [ ]:
def preprocess_data(df):
    # Drop 'Time' (not predictive), keep 'Amount'
    df = df.drop('Time', axis=1)

    # Scale 'Amount' (V1-V28 are already PCA-scaled)
    scaler = StandardScaler()
    df['Amount'] = scaler.fit_transform(df[['Amount']])

    X = df.drop('Class', axis=1)
    y = df['Class']
    return X, y, scaler

def split_data(X, y):
    return train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

X, y, scaler = preprocess_data(df)
X_train, X_test, y_train, y_test = split_data(X, y)

print(f'Training set:   {X_train.shape}')
print(f'Test set:       {X_test.shape}')

### 3. Handle Class Imbalance with SMOTE

In [ ]:
# SMOTE: Synthetic Minority Over-sampling Technique
# Applied ONLY on training data (never on test data!)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('Before SMOTE:', dict(pd.Series(y_train).value_counts()))
print('After SMOTE: ', dict(pd.Series(y_train_res).value_counts()))

### 4. Save Preprocessed Data

In [ ]:
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)

# Save splits
joblib.dump((X_train_res, X_test, y_train_res, y_test), 'data/train_test_split.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

print('✅ Preprocessed data saved to data/ and models/')